# RDD Fundamentals

An **RDD (Resilient Distributed Dataset)** is the foundational data abstraction in Spark. Three defining properties make it unique:

| Property | Meaning |
|----------|---------|
| **Resilient** | Fault-tolerant — lost partitions can be recomputed from the lineage graph |
| **Distributed** | Data is split into partitions spread across worker nodes |
| **Dataset** | A collection of records; each record has a known type (e.g., `int`, `str`, `tuple`) |

RDDs were Spark's original API (Spark 1.x). Today DataFrames are preferred for structured data, but RDDs remain essential for:
- Unstructured or semi-structured data
- Fine-grained control over partitioning and execution
- Understanding what DataFrames do under the hood

In this notebook we will:
- Create RDDs and apply transformations
- Understand lazy evaluation and the lineage graph
- Build a classic word-count pipeline
- Explore the most common RDD actions

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("02-RDD-Fundamentals")
    .getOrCreate()
)

# sc is the SparkContext — the gateway to the RDD API
sc = spark.sparkContext

print("SparkContext ready. Master:", sc.master)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 10:45:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkContext ready. Master: spark://spark-master:7077


## Transformations vs Actions — Lazy Evaluation

Spark operations fall into two categories:

### Transformations
- Return a **new RDD** derived from the parent
- **Lazy** — they only record what needs to be done; no computation happens immediately
- Examples: `.map()`, `.filter()`, `.flatMap()`, `.groupByKey()`, `.reduceByKey()`

### Actions
- Trigger **actual computation** and return a result to the driver (or write to storage)
- **Eager** — calling an action kicks off a Spark job
- Examples: `.collect()`, `.count()`, `.take(n)`, `.first()`, `.reduce()`, `.saveAsTextFile()`

```
RDD_A  --[map]--> RDD_B  --[filter]--> RDD_C  --[collect()]--> Python list
         lazy       lazy       lazy        TRIGGERS JOB
```

> **Why lazy?** Spark can see the *entire* pipeline before executing it, which lets the optimizer (Catalyst / DAG Scheduler) reorder, combine, and prune steps for efficiency — similar to how a SQL query planner works.

In [ ]:
# Start with a simple numeric RDD
numbers = sc.parallelize(range(1, 21))   # 1 to 20

# TRANSFORMATION: .map() — apply a function to every element
# Returns a new RDD; no computation yet
squared = numbers.map(lambda x: x ** 2)

# TRANSFORMATION: .filter() — keep only elements matching a predicate
# Still no computation — just extends the lineage
even_squares = squared.filter(lambda x: x % 2 == 0)

# TRANSFORMATION: .flatMap() — like map but flattens one level of nesting
# e.g., [[1,2],[3,4]] -> [1,2,3,4]
words_rdd = sc.parallelize(["hello world", "spark is fast"])
tokens = words_rdd.flatMap(lambda line: line.split())   # still lazy

# At this point NO task has been submitted to the cluster.
# We can inspect the lineage string to see the recorded plan:
print("Lineage of even_squares:")
print(even_squares.toDebugString().decode())

In [ ]:
# ACTION: .collect() — THIS triggers the job on the cluster
# Spark will now:
#   1. Build a DAG from the recorded transformations
#   2. Split the DAG into stages at shuffle boundaries
#   3. Send tasks to executors
#   4. Collect the results back to the driver

result_even_squares = even_squares.collect()
print("Even squares (1-20):", result_even_squares)

result_tokens = tokens.collect()
print("Tokens from flatMap :", result_tokens)

## Lineage Graph & Fault Tolerance

Every RDD remembers *how it was created* — this chain of parent RDDs and transformations is called the **lineage graph** (or DAG — Directed Acyclic Graph).

```
parallelize(1..20)
      │
      ▼
  .map(x²)          ← RDD_squared
      │
      ▼
  .filter(even)     ← RDD_even_squares
      │
      ▼
  .collect()        ← ACTION: executes everything above
```

If a worker node crashes mid-job and a partition is lost, Spark does **not** need a replica. It re-executes only the failed partition's lineage on another node. This is fundamentally different from systems like Hadoop HDFS, which rely on data replication for fault tolerance.

In [ ]:
# ── Word Count ─────────────────────────────────────────────────────────────
# The "Hello World" of distributed computing — count how often each word
# appears across a collection of text lines.

sentences = [
    "the quick brown fox jumps over the lazy dog",
    "the fox ran quickly over the hill",
    "a quick brown dog outpaced a lazy fox",
    "the dog barked at the fox",
]

# Step 1: distribute the list across the cluster
lines_rdd = sc.parallelize(sentences)

# Step 2: split each line into individual words (flatMap flattens the list of lists)
words = lines_rdd.flatMap(lambda line: line.split())

# Step 3: emit (word, 1) pairs — one pair per word occurrence
pairs = words.map(lambda word: (word, 1))

# Step 4: sum the 1s for each key — this causes a SHUFFLE (data moves between nodes)
word_counts = pairs.reduceByKey(lambda a, b: a + b)

# Step 5: sort descending by count for readability
sorted_counts = word_counts.sortBy(lambda kv: kv[1], ascending=False)

# Step 6: ACTION — execute the full pipeline and bring results to driver
results = sorted_counts.collect()

print("Word counts (sorted by frequency):")
for word, count in results:
    print(f"  {word:<15} {count}")

In [ ]:
# ── Common RDD Actions ─────────────────────────────────────────────────────
# Each call below triggers a separate Spark job.

nums = sc.parallelize([4, 8, 15, 16, 23, 42, 7, 3, 11])

# .count() — total number of elements (one fast job, no data returned)
print("count()         :", nums.count())

# .take(n) — return the first n elements WITHOUT sorting
# (faster than collect() for large RDDs — stops as soon as n elements found)
print("take(5)         :", nums.take(5))

# .first() — shorthand for take(1)[0]
print("first()         :", nums.first())

# .reduce() — apply a commutative+associative function across all elements
# Works in parallel: each partition reduces independently, then results merge
total = nums.reduce(lambda a, b: a + b)
print("reduce(sum)     :", total)

# .countByValue() — returns a dict of {value: count} — useful for small cardinality
small_rdd = sc.parallelize(["a", "b", "a", "c", "b", "a"])
print("countByValue()  :", dict(small_rdd.countByValue()))

In [3]:
# Release cluster resources
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
